In [ ]:
#Cell 0
%uv pip install librosa matplotlib yt-dlp demucs faster-whisper torchaudio soundfile tensorboard pyyaml
import os
for d in ["/workspace/dataset/raw", "/workspace/dataset/clean", "/workspace/dataset/chunks", "/workspace/dataset/plots"]:
    os.makedirs(d, exist_ok=True)

In [ ]:
#Cell 1
#Cell 1
import subprocess

WORKSPACE_DIR = "/workspace/dataset"
RAW_DIR = f"{WORKSPACE_DIR}/raw"
CLEAN_DIR = f"{WORKSPACE_DIR}/clean"
FULL_AUDIO_PATH = f"{RAW_DIR}/full_audio.mp3"
CHUNK_AUDIO_PATH = f"{RAW_DIR}/source_chunk.wav"

subprocess.run(["ffmpeg", "-y", "-i", FULL_AUDIO_PATH, "-ss", "00:15:00", "-to", "00:45:00", "-ar", "32000", "-ac", "1", "-c:a", "pcm_s16le", CHUNK_AUDIO_PATH], check=True)
subprocess.run(["demucs", "-n", "htdemucs", "--two-stems=vocals", "-d", "cuda", "--out", CLEAN_DIR, CHUNK_AUDIO_PATH], check=True)

In [ ]:
#Cell 2
import os
import torch
from faster_whisper import WhisperModel

WORKSPACE_DIR = "/workspace/dataset"
CLEAN_DIR = f"{WORKSPACE_DIR}/clean"
VOCAL_PATH = f"{CLEAN_DIR}/htdemucs/source_chunk/vocals.wav"
CHUNK_DIR = f"{WORKSPACE_DIR}/chunks"
LIST_TXT = f"{WORKSPACE_DIR}/list.txt"
EXP_NAME = "girl_high"

os.makedirs(CHUNK_DIR, exist_ok=True)

for f in os.listdir(CHUNK_DIR):
    os.remove(os.path.join(CHUNK_DIR, f))
print("--> BẮT ĐẦU VAD (Nhận diện giọng nói)...")
model_vad, utils = torch.hub.load(repo_or_dir='snakers4/silero-vad', model='silero_vad')
get_speech_timestamps, save_audio, read_audio, _, _ = utils

wav = read_audio(VOCAL_PATH, sampling_rate=16000)
speech_timestamps = get_speech_timestamps(wav, model_vad, sampling_rate=16000, min_speech_duration_ms=500, max_speech_duration_s=10, min_silence_duration_ms=500)
print(f"--> VAD TÌM THẤY VÀ CẮT ĐƯỢC: {len(speech_timestamps)} ĐOẠN ÂM THANH.")
if len(speech_timestamps) == 0:
    raise ValueError("VAD ĐIẾC! KHÔNG TÌM THẤY CÂU NÓI NÀO ĐẠT CHUẨN. KIỂM TRA LẠI AUDIO.")

for i, ts in enumerate(speech_timestamps):
    save_audio(f"{CHUNK_DIR}/chunk_{i:04d}.wav", wav[ts['start']: ts['end']], 16000)

print("--> BẮT ĐẦU WHISPER PHIÊN DỊCH TIẾNG ANH...")
model_whisper = WhisperModel("large-v3", device="cuda", compute_type="float16")
wav_files = sorted([f for f in os.listdir(CHUNK_DIR) if f.endswith('.wav')])

result_lines = []
for wav_file in wav_files:
    wav_path = f"{CHUNK_DIR}/{wav_file}"
    segments, _ = model_whisper.transcribe(wav_path, language="en", beam_size=5)
    text = "".join([s.text for s in segments]).replace("|", "").strip()
    if text:
        result_lines.append(f"{wav_path}|{EXP_NAME}|EN|{text}")

with open(LIST_TXT, "w", encoding="utf-8") as f:
    f.write("\n".join(result_lines))

print(f"--> HOÀN TẤT! ĐÃ LƯU {len(result_lines)} DÒNG VÀO LIST.TXT")

In [ ]:
#Cell 3
import os
import subprocess
import urllib.request

WORKSPACE_DIR = "/workspace"
REPO_DIR = f"{WORKSPACE_DIR}/GPT-SoVITS"
EXP_NAME = "girl_high"
LIST_TXT = f"{WORKSPACE_DIR}/dataset/list.txt"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "https://github.com/RVC-Boss/GPT-SoVITS.git", REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run(["pip", "install", "-r", "requirements.txt", "pyyaml", "nltk"], check=True)
subprocess.run(["python", "-c", "import nltk; nltk.download('averaged_perceptron_tagger_eng'); nltk.download('averaged_perceptron_tagger'); nltk.download('cmudict')"], check=True)

subprocess.run("""
apt-get update && apt-get install -y git-lfs && git lfs install
mkdir -p GPT_SoVITS/pretrained_models
cd GPT_SoVITS/pretrained_models
if [ ! -d "chinese-roberta-wwm-ext-large" ]; then git clone https://huggingface.co/hfl/chinese-roberta-wwm-ext-large; fi
if [ ! -d "chinese-hubert-base" ]; then git clone https://huggingface.co/TencentGameMate/chinese-hubert-base; fi
""", shell=True, check=True)

s2G_path = f"{REPO_DIR}/GPT_SoVITS/pretrained_models/s2G488k.pth"
if not os.path.exists(s2G_path):
    urllib.request.urlretrieve("https://huggingface.co/lj1995/GPT-SoVITS/resolve/main/s2G488k.pth", s2G_path)

with open(LIST_TXT, "r", encoding="utf-8") as f:
    raw_lines = f.read().splitlines()

valid_lines = []
for line in raw_lines:
    line = line.strip()
    if not line: continue
    if line.count("|") == 3:
        valid_lines.append(line)
    else:
        print(f"[RÁC BỊ LOẠI BỎ]: {line}")

if not valid_lines:
    raise ValueError("KHÔNG CÓ DỮ LIỆU ĐỂ TRÍCH XUẤT. HỆ THỐNG DỪNG KHẨN CẤP.")

with open(LIST_TXT, "w", encoding="utf-8") as f:
    f.write("\n".join(valid_lines))

my_env = os.environ.copy()
my_env["PYTHONPATH"] = f"{REPO_DIR}:{REPO_DIR}/GPT_SoVITS:{my_env.get('PYTHONPATH', '')}"
my_env["opt_dir"] = f"{WORKSPACE_DIR}/logs/{EXP_NAME}"
my_env["is_half"] = "True"
my_env["bert_pretrained_dir"] = f"{REPO_DIR}/GPT_SoVITS/pretrained_models/chinese-roberta-wwm-ext-large"
my_env["cnhubert_base_dir"] = f"{REPO_DIR}/GPT_SoVITS/pretrained_models/chinese-hubert-base"
my_env["pretrained_s2G"] = s2G_path
my_env["s2config_path"] = f"{REPO_DIR}/GPT_SoVITS/configs/s2.json"
my_env["inp_text"] = LIST_TXT
my_env["exp_name"] = EXP_NAME
my_env["i_part"] = "0"
my_env["all_parts"] = "1"

subprocess.run(["python", "GPT_SoVITS/prepare_datasets/1-get-text.py", "--inp_text", LIST_TXT, "--exp_name", EXP_NAME], env=my_env, check=True)
subprocess.run(["python", "GPT_SoVITS/prepare_datasets/2-get-hubert-wav32k.py", "--inp_text", LIST_TXT, "--exp_name", EXP_NAME], env=my_env, check=True)
subprocess.run(["python", "GPT_SoVITS/prepare_datasets/3-get-semantic.py", "--inp_text", LIST_TXT, "--exp_name", EXP_NAME], env=my_env, check=True)

In [ ]:
#Cell 4
import json
import yaml
import subprocess
import os
import glob
import urllib.request

# ═══════════════════════════════════════════════
# CẤU HÌNH - TỰ KHAI BÁO TOÀN BỘ, KHÔNG PHỤ THUỘC CELL KHÁC
# ═══════════════════════════════════════════════
WORKSPACE_DIR = "/workspace"
REPO_DIR      = f"{WORKSPACE_DIR}/GPT-SoVITS"
EXP_NAME      = "girl_high"
EXP_DIR       = f"{WORKSPACE_DIR}/logs/{EXP_NAME}"
WEIGHTS_DIR   = f"{WORKSPACE_DIR}/weights"
LIST_TXT      = f"{WORKSPACE_DIR}/dataset/list.txt"

PRETRAINED_DIR = f"{REPO_DIR}/GPT_SoVITS/pretrained_models"
S2_JSON        = f"{REPO_DIR}/GPT_SoVITS/configs/s2.json"
S1_YAML        = f"{REPO_DIR}/GPT_SoVITS/configs/s1.yaml"

os.makedirs(EXP_DIR,    exist_ok=True)
os.makedirs(WEIGHTS_DIR, exist_ok=True)
os.chdir(REPO_DIR)

# ═══════════════════════════════════════════════
# ENV - SET ĐẦY ĐỦ CÁC BIẾN MÀ SCRIPT ĐỌC TỪ os.environ
# ═══════════════════════════════════════════════
my_env = os.environ.copy()
my_env["PYTHONPATH"]           = f"{REPO_DIR}:{REPO_DIR}/GPT_SoVITS:{my_env.get('PYTHONPATH', '')}"
my_env["version"]              = "v1"
my_env["is_half"]              = "True"
my_env["opt_dir"]              = EXP_DIR          # ← FIX LỖI CHÍNH
my_env["exp_name"]             = EXP_NAME
my_env["inp_text"]             = LIST_TXT
my_env["i_part"]               = "0"
my_env["all_parts"]            = "1"
my_env["bert_pretrained_dir"]  = f"{PRETRAINED_DIR}/chinese-roberta-wwm-ext-large"
my_env["cnhubert_base_dir"]    = f"{PRETRAINED_DIR}/chinese-hubert-base"
my_env["pretrained_s2G"]       = f"{PRETRAINED_DIR}/s2G488k.pth"
my_env["s2config_path"]        = S2_JSON

# ═══════════════════════════════════════════════
# BƯỚC 1 - KÉO TRỌNG SỐ NẾU CHƯA CÓ
# ═══════════════════════════════════════════════
print("--> KIỂM TRA VÀ KÉO TRỌNG SỐ...")
models = [
    ("https://huggingface.co/lj1995/GPT-SoVITS/resolve/main/s2G488k.pth",                                           "s2G488k.pth"),
    ("https://huggingface.co/lj1995/GPT-SoVITS/resolve/main/s2D488k.pth",                                           "s2D488k.pth"),
    ("https://huggingface.co/lj1995/GPT-SoVITS/resolve/main/s1bert25hz-2kh-longer-epoch=68e-step=50232.ckpt",       "s1bert25hz-2kh-longer-epoch.ckpt"),
]
for url, name in models:
    path = f"{PRETRAINED_DIR}/{name}"
    if not os.path.exists(path):
        print(f"  Đang tải: {name}")
        urllib.request.urlretrieve(url, path)
    else:
        print(f"  Đã có: {name}")

# ═══════════════════════════════════════════════
# BƯỚC 2 - ĐỒNG BỘ HÓA PHONEMES (gọi trực tiếp bằng path, không dùng -m)
# ═══════════════════════════════════════════════
print("--> ĐỒNG BỘ HÓA TỪ ĐIỂN PHONEMES...")
subprocess.run([
    "python", "GPT_SoVITS/prepare_datasets/1-get-text.py",
    "--inp_text", LIST_TXT,
    "--exp_name", EXP_NAME,
], env=my_env, check=True)

# ═══════════════════════════════════════════════
# BƯỚC 3 - DỌN FILE ĐẦU RA ĐUÔI -0
# ═══════════════════════════════════════════════
print("--> DỌN RÁC ĐUÔI -0...")
for file in glob.glob(f"{EXP_DIR}/*-0.*"):
    new_file = file.replace("-0.", ".")
    if os.path.exists(new_file):
        os.remove(new_file)
    os.rename(file, new_file)
    print(f"  Đổi tên: {os.path.basename(file)} → {os.path.basename(new_file)}")

# ═══════════════════════════════════════════════
# BƯỚC 4 - TRAIN S2 (SoVITS)
# ═══════════════════════════════════════════════
os.makedirs(f"{EXP_DIR}/logs_s2_v1", exist_ok=True)
os.makedirs(f"{EXP_DIR}/logs_s2",    exist_ok=True)

print("--> PATCH s2.json...")
with open(S2_JSON, "r", encoding="utf-8") as f:
    s2_config = json.load(f)

# Patch các field hay bị thiếu
train_cfg = s2_config.setdefault("train", {})
train_cfg.setdefault("gpu_numbers", "0")          # GPU index, "0" cho 1 GPU
train_cfg.setdefault("batch_size", 4)
train_cfg.setdefault("total_epoch", 8)
train_cfg.setdefault("text_low_lr_rate", 0.4)
train_cfg.setdefault("if_save_latest", True)
train_cfg.setdefault("if_save_every_weights", True)
train_cfg.setdefault("save_every_epoch", 4)
train_cfg.setdefault("pretrained_s2G", f"{PRETRAINED_DIR}/s2G488k.pth")
train_cfg.setdefault("pretrained_s2D", f"{PRETRAINED_DIR}/s2D488k.pth")

# Patch các path model
s2_config.setdefault("model", {})
s2_config["model"].setdefault("version", "v1")

# Patch output dir
s2_config.setdefault("s2_ckpt_dir",  EXP_DIR)
s2_config.setdefault("save_weight_dir", WEIGHTS_DIR)
s2_config.setdefault("name", EXP_NAME)
s2_config["data"]["exp_dir"]          = EXP_DIR
s2_config["data"]["training_files"]   = f"{EXP_DIR}/6-name2semantic-0.tsv"
s2_config["data"]["semantic_path"]    = f"{EXP_DIR}/6-name2semantic-0.tsv"
s2_config["data"]["phoneme_path"]     = f"{EXP_DIR}/2-name2text-0.txt"
s2_config["data"]["hubert_path"]      = f"{EXP_DIR}/4-cnhubert"
s2_config["data"]["wav32k_path"]      = f"{EXP_DIR}/5-wav32k"
s2_config["s2_ckpt_dir"] = EXP_DIR
s2_config["train"]["epochs"] = 16
s2_config["train"]["batch_size"] = 8

with open(S2_JSON, "w", encoding="utf-8") as f:
    json.dump(s2_config, f, indent=2, ensure_ascii=False)
print("  s2.json đã được patch xong.")

print("--> BẮT ĐẦU TRAIN S2 (SoVITS)...")
subprocess.run([
    "python", "GPT_SoVITS/s2_train.py",
    "--config", S2_JSON,
], env=my_env, check=True)

# ═══════════════════════════════════════════════
# BƯỚC 5 - TRAIN S1 (GPT)
# ═══════════════════════════════════════════════
os.makedirs(f"{EXP_DIR}/logs_s1", exist_ok=True)
print("--> PATCH s1.yaml...")
with open(S1_YAML, "r", encoding="utf-8") as f:
    s1_config = yaml.safe_load(f)

# Dùng = thay vì setdefault để ghi đè giá trị cũ
s1_config["train"]["seed"]                  = 1234
s1_config["train"]["epochs"]                = 15        # ← script đọc "epochs" không phải "total_epoch"
s1_config["train"]["precision"]             = 16
s1_config["train"]["save_every_n_epoch"]    = 5         # ← script đọc field này
s1_config["train"]["if_save_latest"]        = True
s1_config["train"]["if_save_every_weights"] = True
s1_config["train"]["half_weights_save_dir"] = WEIGHTS_DIR
s1_config["train"]["exp_name"]              = EXP_NAME  # ← field bị thiếu gây lỗi vừa rồi
s1_config["train"]["gpu_numbers"]           = "0"

s1_config["exp_name"]        = EXP_NAME
s1_config["exp_dir"]         = EXP_DIR
s1_config["output_dir"]      = EXP_DIR
s1_config["save_weight_dir"] = WEIGHTS_DIR
s1_config["pretrained_s1"]   = f"{PRETRAINED_DIR}/s1bert25hz-2kh-longer-epoch.ckpt"
s1_config["model"]["n_layer"] = 24
s1_config["train_semantic_path"] = f"{EXP_DIR}/6-name2semantic.tsv"
s1_config["train_phoneme_path"]  = f"{EXP_DIR}/2-name2text.txt"

with open(S1_YAML, "w", encoding="utf-8") as f:
    yaml.dump(s1_config, f, allow_unicode=True)
print("  s1.yaml đã được patch xong.")

print("--> BẮT ĐẦU TRAIN S1 (GPT)...")
subprocess.run([
    "python", "GPT_SoVITS/s1_train.py",
    "--config", S1_YAML,
], env=my_env, check=True)

print("\n✅ HOÀN TẤT TOÀN BỘ QUÁ TRÌNH TRAIN!")

Debug

In [ ]:
import subprocess
result = subprocess.run(["grep", "-n", 'config\["train"\]', "/workspace/GPT-SoVITS/GPT_SoVITS/s1_train.py"], capture_output=True, text=True)
print(result.stdout)

In [ ]:
import shutil

# Xóa checkpoint s2 cũ
shutil.rmtree(f"{EXP_DIR}/logs_s2_v1", ignore_errors=True)
shutil.rmtree(f"{EXP_DIR}/logs_s2",    ignore_errors=True)

# Tạo lại folder trống
os.makedirs(f"{EXP_DIR}/logs_s2_v1", exist_ok=True)
os.makedirs(f"{EXP_DIR}/logs_s2",    exist_ok=True)

print("✅ Đã xóa model cũ, sẵn sàng train lại.")

In [ ]:
import yaml
S1_YAML = "/workspace/GPT-SoVITS/GPT_SoVITS/configs/s1.yaml"
with open(S1_YAML) as f:
    print(f.read())

In [ ]:
import json
S2_JSON = "/workspace/GPT-SoVITS/GPT_SoVITS/configs/s2.json"
with open(S2_JSON) as f:
    print(json.dumps(json.load(f), indent=2))